# Introduction to FEniCS
In the previous lecture you learnt how to assemble and solve a system of equations for a Poisson equation on line subdivided into $N$ equally large line segments.
However, in the real world, we will rarely work on so simple geometries, and will therefore find a higher level abstraction for representing these kinds of problems.

In this and the following lectures, we will use [FEniCS](https://fenicsproject.org/download/), a collection of software designed for the automated solution of partial differential equations using the finite element method. It handles a lot of the busywork involved for you, and more or less automates everything except the mathematical derivation of the problem (strong  and weak formulations). 
The community around the FEniCS project is very active and a lot of useful information and demo files can be found at the FEniCS Discourse (https://fenicsproject.discourse.group/). I recommend each of you to download the free tutorial book "Solving PDE's in Python - The FEniCS Tutorial I" (https://fenicsproject.org/tutorial/). It covers most of the Python functionality of FEniCS and is a good starting for any one who wants to use FEniCS.

Further tutorials on FEniCS can be found at https://jsdokken.com/dolfinx-tutorial/, https://jsdokken.com/FEniCS-workshop/ and https://docs.fenicsproject.org/dolfinx/v0.10.0.post5/python/demos.html.

## Reimplementing the FEM intro problem on a grid with spatially varying cells
In this demo, we will consider a problem similar to [L01_FEM_intro](./L01_FEM_intro), but we will use a grid where the points are not equally spaced.
We start by importing the modules we require for generating our mesh

In [1]:

from mpi4py import MPI
import numpy as np
import basix

The modules we have import so far is {py:mod}`mpi4py`, a Python module for handling communication across different processes (CPUs) on a system. In this lecture we will not focus on how this works. Furthermore, we import {py:mod}`numpy` for usage of numerical arrays, and {py:mod}`basix` which is a finite element library.
Next, we generate two arrays, one called `nodes` which contains the points of our mesh and `cells` which is a 2D array where the i-th row describes the position of the vertices of cell `i` in `nodes`

In [2]:
cells = np.array([[0,1], [1,2], [2, 3], [3, 4], [4, 6], [6,5]], dtype=np.int64)
nodes = np.array([[0.1], [0.2], [0.4], [0.5],[0.6], [1.], [0.8]], dtype=np.float64)

Furthermore, as each cell is not the same size, we cannot pre-compute the integrals in the matrix `A` for all cells. A key-ingredient in the finite element method is to **pull back** all integrals to a reference cell, meaning that the integral

$$ \int_a^b f(x)~\mathrm{d}x = \int_0^1 f(F(\hat x))\vert b - a \vert~\mathrm{d}\hat x,$$
where $F(\hat x):[0, 1]\mapsto[a, b]=a + \hat x (b-a)$.

More generally, if there exists a mapping: $F:K_{ref}\mapsto K$ then we can write the integral

$$ \int_K f(x)~\mathrm{d}x = \int_{K_{ref}} f(F(\hat x))\vert \mathrm{det} J\vert~\mathrm{d}\hat x,$$

where $J$ is the Jacobian of the mapping $F$, i.e. $J = \frac{\partial F}{\partial \hat x}$. This is generally known as the method of mappings.

We choose to describe any cell $K$ by a finite element, i.e.

$$
F_i(\hat x) = \sum_{k=0}^N \mathbf{p}_{i,k} \phi_k(\hat x)
$$
where $\mathbf{p}_{i,k}$ is the $k$ node of the $i$ th cell, where $\phi_k$ is the $k$-basis function of the chosen space. For straight intervals, we can choose a first order Lagrange element. To be general, we use basix to create this element.
To create a {py:class}`dolfinx.mesh.Mesh` object, we use the function {py:func}`dolfinx.mesh.create_mesh` to collect all the data as a single object.

In [3]:
from dolfinx.mesh import create_mesh
c_el = basix.ufl.element("Lagrange", "interval", 1, shape=(nodes.shape[1], ))
mesh = create_mesh(MPI.COMM_SELF, cells=cells, e=c_el, x=nodes)


### Evaluating a finite element at some points
With the basix element, we can evaluate the basis functions at various points.
We use the {py:meth}`tabulate<basix.finite_element.FiniteElement.tabulate>` function to get the $k$-th order derivatives of the basis function.

In [4]:
points = np.array([[0],[0.4],[1]]) # Pick some points in the reference interval
values = c_el.tabulate(0, points) # Tabulate the basis functions (0th order derivatives) at these points
for val, point in zip(values, points):
    print(f"Basis functions at {point}: {val}")

Basis functions at [0.]: [[[ 1.00000000e+00 -3.82105486e-17]]

 [[ 6.00000000e-01  4.00000000e-01]]

 [[ 5.79375858e-17  1.00000000e+00]]]


## Exercise

```{exercise} Compute physical coordiantes
:label: l12-mapping-F
Compute the coordinate of the midpoint of each cell by using the mapping $F_i$
```

Go to {ref}`solution <sol-l12-mapping-F>`.


```{exercise} Compute physical coordinates of a tetrahedron
:label: l12-mapping-tetra-F
Define a tetrahedron by 4 vertices, compute from a reference tetrahedron to physical space.
```

```{dropdown} Hint
To get some points within a reference tetrahedron, you can use {py:func}`basix.create_lattice`, for instance 
```python
reference_points = basix.create_lattice(
    basix.CellType.tetrahedron, 8, basix.LatticeType.gll, exterior=False, method=basix.LatticeSimplexMethod.warp
)
```
Go to {ref}`solution <sol-l12-mapping-tetra-F>`.
```{exercise} Compute the determinant of the Jacobian
:label: l12-mapping-detJ
Create a function that compute sthe determinant of the Jacobian for various cells.
```
Go to {ref}`solution <sol-l12-mapping-detJ>`.

## Numerical integration
Why do we want to be able to evaluate the Jacobian and determinant of Jacobian at discrete points?
Recall that we had this integral

$$ \int_K f(x)~\mathrm{d}x = \int_{K_{ref}} f(F(\hat x))\vert \mathrm{det} J\vert~\mathrm{d}\hat x,$$

which we now want to compute numerically. To do this we choose an appropriate [quadrature rule](https://quadraturerules.org/), i.e. a set of points $\mathbf{q}_i$ and weights $w_i$ such that:

$$ \int_K f(x)~\mathrm{d}x \approx \sum_i w_i f(\mathbf{q}_i).$$

As we have moved the integral back to the reference element $K_{ref}$, we can choose from well known quadrature rules on this element.
We will use {py:func}`basix.make_quadrature(cell, degree)<basix.make_quadrature>` to get quadrature rules that are exact for polynomials of a given degree. 

In [5]:
q_points, q_weights = basix.make_quadrature(basix.CellType.triangle, 4)
print(f"{q_weights=}")
print(f"{q_points=}")

q_weights=array([0.05497587, 0.05497587, 0.05497587, 0.11169079, 0.11169079,
       0.11169079])
q_points=array([[0.81684757, 0.09157621],
       [0.09157621, 0.81684757],
       [0.09157621, 0.09157621],
       [0.10810302, 0.44594849],
       [0.44594849, 0.10810302],
       [0.44594849, 0.44594849]])


For the Poisson problem we considered in the FEM intro, we have the integral

$$
\int_K \nabla \phi_i \cdot \nabla \phi_j ~\mathrm{d}x
$$

which when pulled back to the reference element can be written as

$$
\int_{K_{ref}} J^{-T} \hat\nabla \phi_i(\hat x) \cdot J^{-T} \hat\nabla \phi_j(\hat x)\vert \mathrm{det}J\vert ~\mathrm{d}x
$$

where $\hat \nabla = \big[\frac{\partial }{\partial \hat x_0}, \dots \frac{\partial }{\partial \hat x_N}]^T$ is the reference gradient, which relates through the physical gradient by $\nabla G=J^T \hat \nabla G$.

## Exercise

```{exercise} Compute local stiffness matrix for a single element
:label: l12-stiffness
Create a code for assembling the local stiffness matrix on a triangular cell.
```

Go to {ref}`solution <sol-l12-stiffness>`.


## Assembling over multiple cells
Now that we know how to compute the local tensor for a single cell, we are ready to compute it over multiple cells.
We create a unit square consisting of triangular elements to illustrate how to use our local assembly within DOLFINx.

In [57]:
from mpi4py import MPI
from petsc4py import PETSc
import numpy as np
import dolfinx

mesh = dolfinx.mesh.create_unit_square(MPI.COMM_SELF, 10, 4, cell_type=dolfinx.mesh.CellType.triangle)
space_degree = 2
V = dolfinx.fem.functionspace(mesh, ("Lagrange", space_degree))

quadrature_degree = 2 * (space_degree-1)
q_points, q_weights = basix.make_quadrature(mesh.basix_cell(), quadrature_degree)

As we know that the global matrix will be sparse (as each basis function has local support), we will create a sparse matrix (i.e. a matrix that doesn't store all entries). We do this by using the {py:class}`SparsityPattern<dolfinx.cpp.la.SparsityPattern>` class.
For each cell in our grid, we will use the degree of freedom map (dofmap) to insert all potential non-zero entries that we can encounter.

In [67]:
tdim = mesh.topology.dim
num_cells = mesh.topology.index_map(tdim).size_local
index_map = V.dofmap.index_map
block_size = V.dofmap.index_map_bs
pattern = dolfinx.cpp.la.SparsityPattern(mesh.comm, [index_map, index_map], [block_size, block_size])
for i in range(num_cells):
    local_dofs = V.dofmap.cell_dofs(i)
    pattern.insert(local_dofs, local_dofs)
pattern.finalize()

Next, we create a {py:class}`PETSc-matrix<petsc4py.PETSc.Mat>` which will store the assembled values and set all entries to be initally zero.

In [59]:
A = dolfinx.cpp.la.petsc.create_matrix(mesh.comm, pattern)
A.zeroEntries()

Next, we create a function for computing the Jacobian and one to assemble the local stiffness matrix

In [60]:
def compute_Jacobian(mesh, nodes, reference_points):
    # Tabulate basis function for the coordinate element
    el = mesh.ufl_domain().ufl_coordinate_element().sub_elements[0]
 
    basis_derivatives = el.tabulate(1, reference_points)[1:]
 
    # Compute the basis derivatives in each direction in reference space
    tdim = el.cell.topological_dimension() # Dimension in reference space
    gdim = nodes.shape[1] # Dimension in physical space
    dphi_dxi = []
    for j in range(tdim):
        dphi_dxi.append(basis_derivatives[j] @ nodes)
  
    # The Jacobian is a (gdim, tdim) tensor at each point
    num_points = reference_points.shape[0]
    return np.transpose(np.hstack(dphi_dxi).reshape(num_points, tdim, gdim), (0, 2, 1))

def compute_local_stiffness_matrix(V, nodes, reference_points, q_weights):
    el = V.ufl_element()
    dphidhatx = el.tabulate(1, q_points)[1:] # Shape is (tdim, num_q_points, num_dofs)
    num_basis_funcs = el.dim
    # Create local (num_dofs, num_dofs) matrix
    A_local = np.zeros((num_basis_funcs, num_basis_funcs))
    # Compute the jacobian, its inverse and determinant at quadrature points
    jacobian = compute_Jacobian(V.mesh, nodes, reference_points)
    jacobian_inv = np.linalg.inv(jacobian) # (num_points, tdim, gdim)
    jacobian_inv_T = np.transpose(jacobian_inv, (0, 2, 1)) # (num_points, gdim, tdim)
    detJ = np.abs(np.linalg.det(jacobian))
    # Insert into local tensor
    for i in range(num_basis_funcs):
        for j in range(num_basis_funcs):
            dphidx = np.einsum("ijk,kil->ijl", jacobian_inv_T, dphidhatx) # (num_q_points, gdim, num_dofs)
            grad_product = np.einsum("ij,ij->i", dphidx[:,:,i], dphidx[:,:,j]) # (num_q_points)
            A_local[i, j] += np.sum(q_weights * grad_product * detJ)
    return A_local

Next, we loop over each cell of the mesh, extract the mesh nodes for that cell and assemble the local matrix

In [61]:
for i in range(num_cells):    
    # Extract cell nodal coordinates
    cell_node_indices = mesh.geometry.dofmap[i]
    local_mesh_geometry = mesh.geometry.x[cell_node_indices, :mesh.geometry.dim]
    A_loc = compute_local_stiffness_matrix(V, local_mesh_geometry, q_points, q_weights)    
    # Insert local matrix into global matrix
    cell_dofs = V.dofmap.cell_dofs(i)
    A.setValuesLocal(cell_dofs, cell_dofs, A_loc, PETSc.InsertMode.ADD_VALUES)
# Accumulate contributions from all processors
A.assemble()

All of this might seem rather cumbersome, as there are tons of different PDEs that exist, and one would not like to compute all these local matrices by hand. The good news is that this can be automated with DOLFINx using the Unified Form Language.

In [66]:
import ufl
from dolfinx.fem.petsc import assemble_matrix
u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)
A_ufl = assemble_matrix(dolfinx.fem.form(ufl.inner(ufl.grad(u), ufl.grad(v))*ufl.dx))
A_ufl.assemble()

if MPI.COMM_WORLD.size == 0:
    np.testing.assert_allclose(A_ufl.to_dense(), A[:,:])

D = A - A_ufl
PETSc.Sys.Print(f"{A.norm(2)=:.5e}, {A_ufl.norm(2)=:.5e}, {D.norm(2)=:.5e}")
assert np.isclose(D.norm(2), 0)

A.norm(2)=1.06529e+02, A_ufl.norm(2)=1.06529e+02, D.norm(2)=6.04322e-14
